# Simulated Chromosome 
March 19, 2023.  Tutorial written by Ben Ruben (benruben@g.harvard.edu)

In this tutorial, we will demonstrate how to simulate a constant-force and constant-distance pulling of mitotic chromosomes using the OpenMiChroM library.  These methods were used in the paper "Structural Reorganization and Relaxation Dynamics of Axially Stressed Chromosomes," published in the Biophysical Journal (DOI:https://doi.org/10.1016/j.bpj.2023.03.029).  This tutorial contains three parts:

## Part 0: Pre-Processing and Auxiliary Functions
This part does some pre-processing of input files and checks whether the polymer remains within the domain of its bonded potential.

## Part 1: Constant-Force Pulling
1. Load a polymer from a condensed starting conformation and define the Homopolymer, Ideal Chromosome, and Type-Type potentials
2. Define and add the "Pin" and "Slide" pulling potentials and equilibrate.
3. Define a constant pulling forces which act on the centers of mass of the "pull groups" defined as the first and last 50 beads of the polymer.
4. Apply constant pulling force and measure end-to-end distance under force.  Save pulled structures for following CD simulations
5. Release constant force and measure extension as the chromosome retracts.

## Part 2: Constant-Distance Pulling
In this section, we demonstrate a constant-distance pulling experiment which measured the force-extension curve of the chromosome.  Frames generated in the constant-force pulling simulation are used as starting structures for simulations.  After the simulations, we plot a rough force-extension curve.

## Run modes and temperature

The default runs the longer teaching protocol. Set `OPENMICHROM_TUTORIAL_MODE=fast` for a CPU smoke test of both pulling experiments; it is too short to measure a force–extension curve. Set `OPENMICHROM_TUTORIAL_PLATFORM` to select another OpenMM platform.

Both modes use $k_BT/\epsilon=1$ (the OpenMiChroM default). The original notebook passed `temperature=120`; the API takes reduced units, not kelvin. That historical setting can drive bonds beyond the finite-extension limit before pulling begins. The current example illustrates the protocol, not a reproduction of the published force–extension curve.

Each execution writes its trajectory, energies and plots to a new `pulling-run-*` directory. The analysis below reads the results of that run.

In [ ]:
import os
from pathlib import Path
from tempfile import mkdtemp

import numpy as np

# Find bundled inputs from either the notebook folder or the repository root.
if not Path('input/DT40_chr7.eigen').is_file():
    for root in (Path.cwd(), *Path.cwd().parents):
        tutorial_dir = root / 'Tutorials/Chromosome_Pulling_Tutorial'
        if (tutorial_dir / 'input/DT40_chr7.eigen').is_file():
            os.chdir(tutorial_dir)
            break

TUTORIAL_FAST = os.environ.get("OPENMICHROM_TUTORIAL_MODE", "full").lower() == "fast"
TUTORIAL_PLATFORM = os.environ.get("OPENMICHROM_TUTORIAL_PLATFORM", "CPU")
np.random.seed(2026)
print(f"OpenMiChroM tutorial mode: {'fast smoke test' if TUTORIAL_FAST else 'longer teaching run'}")
print(f"OpenMM platform: {TUTORIAL_PLATFORM}")


In [ ]:
from OpenMiChroM.ChromDynamics import MiChroM
from OpenMiChroM import CndbTools
from openmm import unit
import matplotlib.pyplot as plt

run_dir = Path(mkdtemp(prefix="pulling-run-", dir=Path.cwd()))
print(f"Results: {run_dir}")
plt.rcParams.update({'font.size': 16, 'figure.figsize': [6.0, 5.0]})

# Part 0: Setting Up Input Files

The example models chicken DT40 chromosome 7 in prometaphase, 15 minutes after release from G2 arrest. For the Hi-C time course, see [Gibcus et al. (2018)](https://doi.org/10.1126/science.aao6135); for modeling the transition into mitosis, see [Contessoto et al. (2025)](https://doi.org/10.1073/pnas.2425225122). The pulling protocol itself is from [Ruben et al. (2023)](https://doi.org/10.1016/j.bpj.2023.03.029).

The two inputs used below are:
- `DT40_chr7.eigen` contains the A/B annotation for each locus.
- `lambdas_ic_15m` contains the ideal chromosome interaction parameters for the mitotic DT40 chromosome.

We will create two additional input files used to set up the simulations.  These files are already included with this tutorial, but the code to generate them is displayed for pedagogical purposes. The first is a text file which contains the type labels, constructed from the .eigen file:

In [ ]:
# The bundled .eigen file contains bead indices and existing type labels.
annotations = np.loadtxt("input/DT40_chr7.eigen", dtype=str, ndmin=2)
types = annotations[:, 1]
assert np.isin(types, ["A1", "B1"]).all()
sequence_path = run_dir / "DT40_chr7.txt"
sequence_path.write_text("".join(f"{i}\t{kind}\n" for i, kind in enumerate(types, 1)))
np.testing.assert_array_equal(np.loadtxt(sequence_path, dtype=str)[:, 1], types)
print("\n".join(sequence_path.read_text().splitlines()[:10]))

Next, we create a text file containing a table of the type-type interaction parameters:

In [ ]:
types_path = run_dir / "lambdas_types_15m"
types_path.write_text("A1,B1\n-1.644455115938274231e-02,-2.043214285797510181e-03\n-2.043214285797510181e-03,-1.512813001184262077e-02\n")
print(types_path.read_text())

In [ ]:
def checked_positions(sim):
    """Return coordinates only while energies and all FENE bonds are physical."""
    state = sim.context.getState(getPositions=True, getEnergy=True)
    xyz = state.getPositions(asNumpy=True).value_in_unit(unit.nanometer)
    energies = [energy.value_in_unit(unit.kilojoule_per_mole) for energy in
                (state.getPotentialEnergy(), state.getKineticEnergy())]
    bonds = np.linalg.norm(np.diff(xyz, axis=0), axis=1)
    # This example has a single linear chain; every adjacent pair is bonded.
    if not np.isfinite(xyz).all() or not np.isfinite(energies).all() or np.any(bonds >= sim.context.getParameter("fr0")):
        raise RuntimeError(f"Unstable trajectory at step {sim.simulation.currentStep}: "
                           "non-finite energy/coordinates or a bond at the FENE extension limit.")
    return xyz

# Part 1: Constant-Force Simulation (Stress-Relaxation Experiment)

## Simulation Parameters:

In [ ]:
force = 5  # Applied to the centers of the two pulling groups.
temperature = 1.0  # kBT / epsilon, in both experiments
platform = TUTORIAL_PLATFORM

bs = 10  # Integration steps per end-to-end measurement.
numBlocks_nat = 2 if TUTORIAL_FAST else 2_000
numBlocks_pull = 3 if TUTORIAL_FAST else 10_000
numBlocks_release = 2 if TUTORIAL_FAST else 2_000
blocksPerFrame = 1 if TUTORIAL_FAST else 150  # Spacing of restarts during pulling.
report_interval = bs if TUTORIAL_FAST else 500
restart_steps = []

## Function that retrieves pull coordinate:

In [ ]:
#This function retrieves the pull coordinate, defined as the distance between the centers of mass of the pull groups, along the x-axis.
def getPCoord(group1, group2, positions):
    first_centroid = np.mean(positions[group1], axis=0) 
    second_centroid = np.mean(positions[group2], axis=0)

    ## calculate r0 distance between groups

    return np.sqrt( (first_centroid[0] - second_centroid[0])**2 )

## Functions to create pin, slide, and pull forces

We use center of mass pulling actng on the pull groups, defined as the first and last 50 beads in the polymer:

\begin{equation} \label{COMs}
\begin{split}
\mathbf{R}_{left} = &(x_l, y_l, z_l) = \frac{1}{50}\sum_{i=1}^{50} \mathbf{r}_i \\
\mathbf{R}_{right} = &(x_r, y_r, z_r) = \frac{1}{50}\sum_{i=N-49}^{N} \mathbf{r}_i
\end{split}
\end{equation}

We define a pull coordinate $\xi = x_r - x_l$, as the chromosome's end-to-end x-distance.  In chromosome pulling experiments, the ends of the chromosome are held by micropipettes and one is moved along a linear track.  To mimic this process, we introduce orientation constraints:
\begin{equation} \label{PinSlide_Main}
\begin{split}
&\mathbf{U}_{pin} = \frac{1}{2} k_r (x_l^2 + y_l^2 + z_l^2) \\
&\mathbf{U}_{slide} = \frac{1}{2} k_r (y_r^2 + z_r^2)
\end{split}
\end{equation}
$\mathbf{U}_{pin}$ restrains the center of mass of the left pull group to the origin.  $\mathbf{U}_{slide}$ restrains the right pull group to the x-axis, but allows it to slide along the x-axis.  A large value $k_r = 10^5 \epsilon/\sigma^2$ is used so that $\mathbf{R}_{left} \approx 0$ and $y_r, z_r \approx 0$.  We use two methods to subject the simulated chromosomes to axial strain.  In constant-force (CF) pulling, a linear potential is used to apply a constant elongating force to the pull coordinate $\xi$.  In constant-distance (CD) pulling, a harmonic potential with a very strong spring constant is used to constrain $\xi$ to a very small window around a chosen reference distance $\xi_0$.
\begin{equation} \label{CFCDPotentials}
\begin{split}
&\mathbf{U}_{CF} = - F \xi\\
&\mathbf{U}_{CD} = \frac{1}{2} k_p (\xi -\xi_0)^2
\end{split}
\end{equation}
A large value of $k_p = 10^5 \epsilon/\sigma^2$ is chosen so that $\xi \approx \xi_0$ during CD sampling.

These potentials are implemented in the functions in the cell below:

In [ ]:
#This function is used in Constant-Distance pulling experiments.  It defines a harmonic restraining potential on the pull coordinate.
def harmonic_pull_force(group1, group2, r0, kp):
    import openmm

    #r0 = r0 * units.meter * 1e-9
    pullequation = "0.5 * kpull * ((x2-x1)- rp)^2" #Enforces x2>x1

    pullforce = openmm.CustomCentroidBondForce(2, pullequation)

    pullforce.addGlobalParameter('kpull', kp)
    pullforce.addGlobalParameter('rp', r0)
    pullforce.addGroup(group1)
    pullforce.addGroup(group2)
    pullforce.addBond([0,1])
    #pullforce.setForceGroup(8)
    return(pullforce)

#This function is used in Constant-Force pulling.  It applies a constant force between the centers of mass of the pull groups.  Written as a potential energy, this is a linear function of the pull coordinate.
def constant_pull_force(group1, group2, f):
    import openmm

    #Pulls x2 to the right and x1 to the left
    pullequation = "-1*f*(x2-x1)"

    pullforce = openmm.CustomCentroidBondForce(2, pullequation)

    pullforce.addGlobalParameter('f', f)
    pullforce.addGroup(group1)
    pullforce.addGroup(group2)
    pullforce.addBond([0,1])
    return(pullforce)

#This potential energy keeps the left pull group near the origin of the coordinate system.
def pin_force(group1, kpin=100):
    import openmm

    #r0 = r0 * units.meter * 1e-9
    pullequation = "0.5 * kpin * (x1^2+y1^2+z1^2)"

    pullforce = openmm.CustomCentroidBondForce(1, pullequation)

    print(str(kpin))
    pullforce.addGlobalParameter('kpin', kpin)
    pullforce.addGroup(group1)
    pullforce.addBond([0])
    return(pullforce)

#This potential energy keeps the right pull group near the positive x-axis.
def slide_force(group1, x_min=0, kslide = 100):
    import openmm

    pullequation = "0.5 * kslide * ((step(x_min-x1)*((x_min-x1)^2))+y1^2+z1^2)"

    pullforce = openmm.CustomCentroidBondForce(1, pullequation)

    pullforce.addGlobalParameter('kslide', kslide)
    pullforce.addGlobalParameter('x_min', x_min)
    pullforce.addGroup(group1)
    pullforce.addBond([0])
    return(pullforce)

## Sets up simulation and loads initial structure

In [ ]:
##Start OpenMiChroM library
sim = MiChroM(name='sim', temperature=temperature)
sim.setup(platform=platform, integrator="Langevin")

In [ ]:
sim.saveFolder(run_dir)

In [ ]:
# Read the pre-oriented conformation and its chromatin sequence together.
dataToLoad = sim.loadPDB(PDBfiles=["condensed.pdb"], ChromSeq=[str(sequence_path)])

In [ ]:
sim.loadStructure(dataToLoad, center=False)

In [ ]:
#Adds Homopolymer forces(update variable name)
sim.addFENEBonds(kFb=30.0)
sim.addAngles(kA=1.0)
sim.addRepulsiveSoftCore(eCut=4.0)

In [ ]:
sim.addCustomTypes(mu=1.51, rc=2.12, TypesTable=str(types_path))

In [ ]:
#Adds Ideal Chromosome potential for mitotic chromosome
sim.addCustomIC(mu=1.51, rc = 2.12, dinit=3, dend=735, IClist = 'input/lambdas_ic_15m')

### Define pull groups.

In [ ]:
# Determine the pulling groups from the loaded coordinates before creating the
# OpenMM Context; all forces must be registered first.
positions = np.asarray(dataToLoad)
index = list(range(len(positions)))
g1 = index[:50] #Left pull group is the first 50 beads
g2 = index[-50:] #Right pull group is the last 50 beads

first_centroid = np.mean(positions[g1], axis=0) 
print("Location of the left pull group:")
print(first_centroid)

second_centroid = np.mean(positions[g2], axis=0)
print("\nLocation of the right pull group:")
print(second_centroid)

## calculate r0 distance between groups

r0 = np.sqrt( (first_centroid[0] - second_centroid[0])**2)

print("\nInitial distance between groups is r = {}".format(r0))

## WARNING! -- Orient Initial Structures
The initial structure used here has been pre-oriented so that the left pull group is very near to the origin and the right pull group is very near to the positive x-axis.  This is so that the pin and slide potentials do not generate very large forces on the initial structure.  When performing simulations with an arbitrary starting structure, make sure to rotate the coordinates so that it aligns with the restraining potentials.

In [ ]:
pforce_sim = constant_pull_force(g1,g2, 0)#We first set the pull force to 0
sim.forceDict["pulling"] =  pforce_sim

pin_force_sim = pin_force(g1, kpin=100) #here we using x_pos = 0 and kp = 100
sim.forceDict["pin"] = pin_force_sim

# Pin right end to positive x-axis
slide_force_sim = slide_force(g2, kslide=100) #here we using x_pos = 0 and kp = 100
sim.forceDict["slide"] = slide_force_sim

# Create the Context only after every force has been registered.
sim.createSimulation()

In [ ]:
sim.createReporters(outputName="CF_traj", statistics=True, traj=True, trajFormat="cndb",
                    energyComponents=True, interval=report_interval)

In [ ]:
time_record = [0.0]
pcoord = [getPCoord(g1, g2, checked_positions(sim))]

def record_phase(blocks, *, save_restarts=False):
    for block in range(1, blocks + 1):
        sim.run(nsteps=bs, report=False)
        step = sim.simulation.currentStep
        xyz = checked_positions(sim)
        time_record.append(sim.timeStep * step)
        pcoord.append(getPCoord(g1, g2, xyz))
        if save_restarts and block % blocksPerFrame == 0:
            restart_steps.append(step)

## Native Simulation
We first simulate without adding a pulling force.

In [ ]:
sim.forceDict

In [ ]:
record_phase(numBlocks_nat)

In [ ]:
# The last unforced frame is the first restart for constant-distance pulling.
restart_steps.append(sim.simulation.currentStep)

## Constant Force

In [ ]:
#Update the force parameter to the specified pulling force.
sim.context.setParameter('f', force)

In [ ]:
assert sim.context.getParameter('f') == force

In [ ]:
record_phase(numBlocks_pull, save_restarts=True)

## Release Simulation

In [ ]:
sim.context.setParameter("f", 0)

In [ ]:
record_phase(numBlocks_release)

In [ ]:
# Finalize the CNDB trajectory before opening it for analysis.
for reporter in sim.simulation.reporters:
    if hasattr(reporter, "close"):
        reporter.close()

In [ ]:
coordinfo = np.column_stack([time_record, pcoord])
assert np.all(np.diff(coordinfo[:, 0]) > 0)
np.savetxt(run_dir / "CF_Pull_Coord.txt", coordinfo, header="time_tau extension_sigma")

# Plot Pull Coordinate
We may now plot the end-to-end distance as a function of time during the constant-force experiment.  The vertical lines indicate when force is applied and released.

In [ ]:
all_times = np.asarray(time_record)
all_pcoord = pcoord

fig, ax = plt.subplots()
ax.plot(all_times, all_pcoord)
ax.set_xlabel(r"Time ($\tau$)")
ax.set_ylabel(r"End to End Distance ($\sigma$)")
ax.set_title('Force = '+ str(force) + r" $\epsilon/\sigma$")
ax.axvline(x=all_times[numBlocks_nat], ls='--', color = 'r')
ax.axvline(x = all_times[numBlocks_nat+numBlocks_pull], color = 'b')
fig.savefig(run_dir / "ExtensionTime.png", bbox_inches='tight')

# Part 2: Constant-Distance Pulling

In [ ]:
numDists = min(1, len(restart_steps) - 1) if TUTORIAL_FAST else 5
kpull = 10**5 #A very strong spring constant is used to restrain the end-to-end distance to a small window.
numBlocks_CD = 2 if TUTORIAL_FAST else 3000 #number of blocks to run each CD simulation

xiVals = [] #Array to store pull coordinates used for pulling
forceVals = []#Array to store measured forces

Use the zero-force frame and a selection of stretched conformations as starting points. The CNDB stores unrounded coordinates in the frame order in which the reporter sampled them. Restart steps must coincide with a saved frame.

In [ ]:
with CndbTools.open(run_dir / "CF_traj_0.cndb") as trajectory:
    starts = []
    for step in restart_steps[:numDists + 1]:
        assert step > 0 and step % report_interval == 0
        frame_id = trajectory.frame_ids[step // report_interval - 1]
        xyz = trajectory.xyz(frames=[frame_id])[0]
        np.testing.assert_allclose(getPCoord(g1, g2, xyz), pcoord[step // bs], rtol=1e-5)
        starts.append(xyz)

In [ ]:
for distNum, coords in enumerate(starts):
    
    print('Starting Simulation ' + str(distNum))
    #Sets up a new simulation
    sim = MiChroM(name='sim', temperature=temperature)
    sim.setup(platform=platform, integrator="Langevin")
    
    #Folder to save outputs
    sim.saveFolder(run_dir / f"constant_distance_{distNum}")
    
    # Register the same bead types and start from the selected CNDB frame.
    sim.loadPDB(PDBfiles=["condensed.pdb"], ChromSeq=[str(sequence_path)])
    sim.loadStructure(coords, center=False)

    #Adds Homopolymer forces
    sim.addFENEBonds(kFb=30.0)
    sim.addAngles(kA=1.0)
    sim.addRepulsiveSoftCore(eCut=4.0)

    #Adds type-type potential for mitotic chromosome
    sim.addCustomTypes(mu=1.51, rc = 2.12, TypesTable=str(types_path))

    #Adds Ideal Chromosome potential for mitotic chromosome
    sim.addCustomIC(mu=1.51, rc = 2.12, dinit=3, dend=735, IClist = 'input/lambdas_ic_15m')
    
    #calculate the initial pull coordinate:
    positions = np.asarray(coords)
    
    first_centroid = np.mean(positions[g1], axis=0) 
    print("Location of the left pull group:")
    print(first_centroid)

    second_centroid = np.mean(positions[g2], axis=0)
    print("\nLocation of the right pull group:")
    print(second_centroid)

    ## calculate r0 distance between groups
    r0 = np.sqrt( (first_centroid[0] - second_centroid[0])**2)  
    print('Initial Distance Is: ' + str(r0))
    
    #Add pulling forcers
    pforce_sim = harmonic_pull_force(g1,g2, r0, kpull)
    sim.forceDict["pulling"] = pforce_sim
    
    pin_force_sim = pin_force(g1, kpin=100) #here we using x_pos = 0 and kp = 100
    sim.forceDict["pin"] = pin_force_sim

    # Pin right end to positive x-axis
    slide_force_sim = slide_force(g2, kslide=100) #here we using x_pos = 0 and kp = 100
    sim.forceDict["slide"] = slide_force_sim

    sim.createSimulation()
    
    pcoord = []
    
    checked_positions(sim)
    for _ in range(numBlocks_CD):
        sim.run(nsteps=bs, report=False)
        pcoord.append(getPCoord(g1, g2, checked_positions(sim)))

    meanDist = np.mean(pcoord)
    meanForce = kpull*(r0-meanDist)
    
    #Add to forces
    forceVals.append(meanForce)
    
    #Add to XiVals:
    xiVals.append(meanDist)

We can now plot a force-extension curve

In [ ]:
fig, ax = plt.subplots()

ax.scatter(xiVals, forceVals)
ax.set_xlabel(r'Extension $(\sigma)$')
ax.set_ylabel(r'Force $(\epsilon/\sigma)$')
fig.savefig(run_dir / "ForceExtension.png", bbox_inches='tight')

The scatter plot pairs each restraint's mean extension with its mean restoring force. In fast mode these are startup measurements, not equilibrium averages. For a quantitative force–extension curve, check convergence at each restraint distance and estimate uncertainty from independent runs.